# Проверка train_relabel_v2

Сравнение с `train_relabel_v1` по исходному номеру строки. Здесь видны изменения в разметке и тексте, проверки на проблемных объявлениях и сравнение char-LR на прежнем `test_gold`. Для своих примеров поменяйте список `examples`.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/processed/train_relabel_v2').is_dir())
sys.path[:0] = [str(ROOT / 'dataset_review'), str(ROOT / 'research/experiments')]
from build_train_dedup_v1 import duplicate_key
from build_train_text_v1 import sha256
from train_relabel_rules import RULES, description_body, is_customer_request, relabel
from train_text_cleaning import SEPARATOR, clean_text

version = ROOT / 'data/processed/train_relabel_v2'
manifest = json.loads((version / 'manifest.json').read_text())
for field in ('source', 'raw_source', 'source_removed_duplicates', 'dataset',
              'label_changes', 'text_changes', 'affected_duplicates'):
    info = manifest[field]
    assert sha256(ROOT / info['path']) == info['sha256'], field

before = pd.read_parquet(ROOT / manifest['source']['path']).set_index('source_line')
after = pd.read_parquet(version / 'train.parquet').set_index('source_line')
labels = pd.read_csv(version / 'label_changes.csv')
texts = pd.read_csv(version / 'text_changes.csv')
assert len(before) == len(after) == manifest['dataset']['records'] == 4749
assert before.index.is_unique and before.index.equals(after.index)
assert before['item_id'].equals(after['item_id'])
assert after['description'].map(duplicate_key).is_unique
assert after['title'].str.strip().ne('').all() and after['description'].str.strip().ne('').all()

def class_ids(entries):
    return frozenset(int(entry['mcId']) for entry in entries)

old_ids = before['found_microcategories'].map(class_ids)
new_ids = after['found_microcategories'].map(class_ids)
actual_label_changes = {
    (n, category, int(category in new_ids[n]))
    for n in after.index for category in old_ids[n] ^ new_ids[n]
}
assert actual_label_changes == set(labels[['source_line', 'mcId', 'new']].itertuples(index=False, name=None))
assert len(labels) == len(actual_label_changes)
assert all(q in after.at[n, 'description'] for n, q in labels[['source_line', 'quote']].itertuples(index=False, name=None))
actual_text_changes = {(n, field) for field in ('title', 'description')
                       for n in after.index[before[field].ne(after[field])]}
assert actual_text_changes == set(texts[['source_line', 'field']].itertuples(index=False, name=None))

print(f'{len(after):,} объявлений, {texts.source_line.nunique()} изменений текста, '
      f'{len(labels)} изменений меток в {labels.source_line.nunique()} объявлениях.')
display(pd.DataFrame.from_dict(manifest['transformations'][1]['by_category'], orient='index')
        .rename(columns={'before': 'Было', 'removed': 'Снято', 'added': 'Добавлено', 'after': 'Стало'}))
display(pd.Series(manifest['transformations'][0]['by_rule'], name='Исправленные поля').to_frame())

4,749 объявлений, 339 изменений текста, 86 изменений меток в 60 объявлениях.


,Было,Снято,Добавлено,Стало
102,1954,22,0,1932
103,2409,3,10,2416
104,2337,2,17,2352
105,2329,2,15,2342
106,2681,15,0,2666


,Исправленные поля
decorative_separators,291
joined_list,36
joined_numbers,10
keycap_spacing,6


## Проверки по номерам строк

Проверяем типовые ошибки, подтверждённые на исходных объявлениях. Для объявлений с ограничением «только в комплексе» берём работы из явно предложенного перечня. Для 106 используется согласованное в проекте правило об одной лишь подготовке.

In [2]:
dune = (323, 492, 1533, 3250, 5325, 5791, 5947, 7121, 7656)
request_examples = (540, 1213, 1571, 2117, 6572, 7052, 7663, 7708, 7916, 8567, 8690, 9064, 9285, 9320)
assert all(106 in old_ids[n] and 106 not in new_ids[n] for n in (*dune, 89, 117))
assert all(not is_customer_request(description_body(after.at[n, 'description'])) for n in request_examples)
assert is_customer_request('Ищу мастера для ремонта квартиры.')
assert is_customer_request(after.at[2594, 'description'])
assert 102 not in new_ids[7663]
assert {103, 104, 105, 106} == new_ids[9094]
assert 'Электрика:' in description_body(after.at[9094, 'description'])
assert 'оклейка обоями' in description_body(after.at[9094, 'description'])
assert not new_ids[4344]
assert {105, 106} == new_ids[7475]
assert all(102 not in new_ids[n] for n in (585, 1305, 1637, 1863, 2607, 7415, 7424, 7475))
assert {104} <= new_ids[2015] and {102, 103, 106} == new_ids[2611]
assert all({103, 104, 105} <= new_ids[n] for n in (2946, 3036, 6461))
assert 104 in new_ids[7557]
assert 103 not in new_ids[4953]
assert 102 not in new_ids[238]
assert 'штукатурка Ремонт' in after.at[238, 'description']
assert '2 электрика' in after.at[5031, 'description']
assert '1 Приеду' in after.at[7413, 'description']
assert 'СтройЛидер' in after.at[96, 'description']
assert '2ГИС' in after.at[231, 'description']
assert '8НеЕ5' in after.at[9652, 'description']
assert not after['description'].map(lambda text: bool(SEPARATOR.search(text))).any()

# Пограничные условия правил.
assert set(relabel('Выполняем черновые и чистовые работы: штукатурка, плитка, сантехника, электрика, обои.', set())) == set(RULES)
assert 103 not in relabel('Выполняем ремонт. Что входит: демонтаж старых материалов (обои, краска, плитка).', set())
assert 103 in relabel('Выполняем ремонт. Укладку плитки отдельно не выполняем, делаем только в комплексе.', set())
assert 103 not in relabel('Выполняем ремонт. Плитку не укладываем.', set())
assert 106 not in relabel('Выполняем работы: штукатурка под покраску, под обои.', set())
assert clean_text('2️⃣электрика') == '2 электрика'
print('Проверки на исходных объявлениях и пограничных формулировках прошли.')

examples = [89, 117, 238, *dune, 4344, 5031, 7663, 9094, 2015, 2611, 2946, 3036, 6461, 7557]
display(pd.DataFrame({'До': [sorted(old_ids[n]) for n in examples],
                      'После': [sorted(new_ids[n]) for n in examples]}, index=examples))
display(labels.groupby(['reason', 'new']).size().rename('Метки').reset_index())
display(labels.loc[labels['source_line'].isin(examples), ['source_line', 'mcId', 'old', 'new', 'quote', 'reason']])

Проверки на исходных объявлениях и пограничных формулировках прошли.


,До,После
89,[106],[]
117,"[104, 106]",[104]
238,"[103, 106]","[103, 106]"
323,"[103, 104, 105, 106]","[103, 104, 105]"
492,"[104, 105, 106]","[104, 105]"
1533,"[103, 104, 105, 106]","[103, 104, 105]"
3250,"[103, 104, 105, 106]","[103, 104, 105]"
5325,"[104, 105, 106]","[104, 105]"
5791,"[103, 104, 105, 106]","[103, 104, 105]"
5947,"[103, 104, 105, 106]","[103, 104, 105]"


,reason,new,Метки
0,В основном описании нет малярных работ или пок...,0,10
1,Выравнивание без указания штукатурки или тольк...,0,2
2,"Действие относится к плитке, в том числе в спи...",1,10
3,Категория упомянута только в поисковом блоке,0,30
4,Покраска или поклейка названа как цель подготовки,0,1
5,Прямое предложение сантехнических работ,1,17
6,Прямое предложение электромонтажных работ,1,15
7,Реставрация покрытия ванны без малярной отделк...,0,1


,source_line,mcId,old,new,quote,reason
2,89,106,1,0,"Если ремонт делается ""НА ВРЕМЯ"" — он обычно об...",В основном описании нет малярных работ или пок...
3,117,106,1,0,РЕСТАВРАЦИЯ ВАНН,Реставрация покрытия ванны без малярной отделк...
5,323,106,1,0,"Ремонт квартир и домов под ключ, Ростов и область",В основном описании нет малярных работ или пок...
10,492,106,1,0,"Ремонт квартир и домов под ключ, Ростов и область",В основном описании нет малярных работ или пок...
18,1533,106,1,0,"Ремонт квартир и домов под ключ, Ростов и область",В основном описании нет малярных работ или пок...
28,2015,104,0,1,сантехнические и электрические работы,Прямое предложение сантехнических работ
32,2611,103,0,1,"Укладка полов(ламинат, плитка","Действие относится к плитке, в том числе в спи..."
34,2946,104,0,1,сантехника,Прямое предложение сантехнических работ
35,2946,105,0,1,электрика,Прямое предложение электромонтажных работ
39,3036,103,0,1,плитка и санузлы,"Действие относится к плитке, в том числе в спи..."


## Сравнение char-LR

Один набор строк и одна модель для обеих версий. Убираем из обоих вариантов объединение точных и нормализованных пересечений с `test_gold`. Разбиение привязано к описаниям v1, поэтому чистка текста не переносит строки между train и dev. `test_gold` используется как есть, пороги по нему не подбираются.

In [3]:
from bench import baseline, canonical, digest, gold_overlap_key, key, matrix, metrics

gold_path = ROOT / 'data/test/test_gold.parquet'
test = [canonical(row, 101) for row in pq.read_table(gold_path).to_pylist()]
normalize = gold_overlap_key()
gold_keys = {key(row['description']) for row in test}
gold_normalized = {normalize(row['description']) for row in test}
excluded = {
    n for frame in (before, after)
    for n, text in frame['description'].items()
    if key(text) in gold_keys or normalize(text) in gold_normalized
}
train = [n for n in before.index if n not in excluded
         and int(digest(key(before.at[n, 'description']))[:8], 16) % 5 != 0]
dev = before.index.difference(excluded | set(train))
print(f'Train {len(train)}, dev {len(dev)}, test {len(test)}, исключено пересечений {len(excluded)}.')
print(f'SHA-256 test_gold {sha256(gold_path)}')

overall, by_category = [], []
for name, frame in [('v1', before), ('v2', after)]:
    train_rows = [canonical(row, 101) for row in frame.loc[train].to_dict('records')]
    predictions = list(baseline(train_rows, test))
    scores = metrics(test, predictions)
    overall.append({'train': name, **{k: scores[k] for k in ('micro_f1', 'exact_match', 'precision', 'recall')}})
    truth, pred = matrix(test), matrix(predictions)
    for pos, (category, score) in enumerate(scores['per_category'].items()):
        if category == 101:
            continue
        by_category.append({'train': name, 'mcId': category, 'F1': score['f1'],
                            'FP': int(np.sum((truth[:, pos] == 0) & (pred[:, pos] == 1))),
                            'FN': int(np.sum((truth[:, pos] == 1) & (pred[:, pos] == 0)))})
display(pd.DataFrame(overall).set_index('train').round(4))
display(pd.DataFrame(by_category).pivot(index='mcId', columns='train', values=['F1', 'FP', 'FN']).round(4))

Train 3817, dev 926, test 80, исключено пересечений 6.
SHA-256 test_gold 5ca036c1e8dc4d7c24b6d1c42689feeecb36958ff1dd411ab1b64ef4f83e9189


,micro_f1,exact_match,precision,recall
train,,,,
v1,0.8829,0.7125,0.9245,0.8448
v2,0.8862,0.7125,0.9250,0.8506


F1           FP        FN     
train      v1      v2   v1   v2   v1   v2
mcId                                     
102    0.8364  0.8519  4.0  3.0  5.0  5.0
103    0.8333  0.8197  3.0  4.0  7.0  7.0
104    0.9189  0.9333  0.0  0.0  6.0  5.0
105    0.9114  0.9114  1.0  1.0  6.0  6.0
106    0.8923  0.8923  4.0  4.0  3.0  3.0